# COMP9312 Project Q1: First Cycle-Causing Edge


Run the cells from top to bottom. Only edit the `FirstCycleEdgeQuery` code cell.

## 1. Code Template
Only edit this cell. Implement `FirstCycleEdgeQuery.query(n, L)`. You may add helper methods or fields inside the class, but do not change the public class name or method signature.

In [1]:
################################################################################
# You can import any Python Standard Library modules.
from typing import List, Optional, Tuple
from collections import deque
################################################################################


class FirstCycleEdgeQuery:
    """
    First cycle-causing edge query. You may add helper methods and fields
    inside this class, but do not change the public signature of query().
    """

    def query(
        self,
        n: int,
        L: List[Tuple[int, int]],
    ) -> Optional[Tuple[Tuple[int, int], List[int]]]:
        """
        Return the first cycle-causing edge and the cycle containing it.

        Parameters
        ----------
        n: The number of vertices in the undirected graph. Vertex IDs range from 0 to n - 1.
        L: The edge insertion stream. Each edge is a tuple (u, v).

        Returns
        -------
        If a first cycle-causing edge (u, v) exists, return:
            [[u, v], [i, ..., j]]
        The order does not matter.

        If no inserted edge creates a cycle, return
            None.
        """
        parent = list(range(n))
        rank = [0] * n
        tree = [[] for _ in range(n)]     # accepted edges = spanning forest

        def find(x):
            root = x
            while parent[root] != root:
                root = parent[root]
            while parent[x] != root:       # path compression
                parent[x], x = root, parent[x]
            return root

        for u, v in L:
            ru, rv = find(u), find(v)
            if ru == rv:
                # u, v already connected -> this edge closes a cycle
                return [[u, v], self._cycle(tree, u, v)]

            tree[u].append(v)
            tree[v].append(u)
            if rank[ru] < rank[rv]:
                ru, rv = rv, ru
            parent[rv] = ru
            if rank[ru] == rank[rv]:
                rank[ru] += 1

        return None

    def _cycle(self, tree, u, v):
        # u and v sit in the same tree; BFS the path between them, then close it.
        prev = {u: u}
        q = deque([u])
        while q:
            x = q.popleft()
            if x == v:
                break
            for y in tree[x]:
                if y not in prev:
                    prev[y] = x
                    q.append(y)

        path = [v]
        while path[-1] != u:
            path.append(prev[path[-1]])
        path.reverse()
        path.append(u)                     # [u, ..., v, u]
        return path

## 2. How to Test Your Code
The following tests use the `FirstCycleEdgeQuery` class defined above. Do not edit this cell. Each test prints the input, your output, the expected output, the running time, and whether the result is correct.

In [2]:
################################################################################
# Do not edit this code cell.
from urllib.request import urlopen, Request
import ast
import re
################################################################################

def fetch_text(url: str) -> str:
    req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(req) as response:
        return response.read().decode("utf-8").strip()

def parse_graph(text: str) -> Tuple[int, List[Tuple[int, int]]]:
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    n = int(lines[0])

    nums = []
    for line in lines[1:]:
        nums.extend(map(int, re.findall(r"-?\d+", line)))

    L = [(nums[i], nums[i + 1]) for i in range(0, len(nums), 2)]

    return n, L

def parse_expected(text: str) -> Optional[Tuple[Tuple[int, int], List[int]]]:
    value = ast.literal_eval(text.strip())

    if isinstance(value, str):
        value = ast.literal_eval(value)

    return value

def normalize_answer(ans: Optional[Tuple[Tuple[int, int], List[int]]]):
    if ans is None:
        return None

    edge, cycle = ans

    normalized_edge = tuple(sorted(edge))

    nodes = cycle[:-1]
    start = nodes.index(min(nodes))
    forward = nodes[start:] + nodes[:start]
    reverse = list(reversed(nodes))
    start_rev = reverse.index(min(reverse))
    backward = reverse[start_rev:] + reverse[:start_rev]

    return normalized_edge, tuple(min(forward, backward)), len(cycle)

def run_tests() -> None:
    base_url = "https://cgi.cse.unsw.edu.au/~cs9312/26T2/project"

    test_ids = range(1, 4)

    all_correct = True

    for i in test_ids:
        print("=" * 80)
        print(f"Test {i}")

        graph_url = f"{base_url}/q1_test_{i}.txt"
        expected_url = f"{base_url}/q1_test_{i}_expected.txt"

        graph_text = fetch_text(graph_url)
        expected_text = fetch_text(expected_url)

        n, L = parse_graph(graph_text)
        expected = parse_expected(expected_text)

        print(f"n = {n}")
        print(f"|L| = {len(L)}")

        solver = FirstCycleEdgeQuery()

        actual = solver.query(n, L)

        ok = (normalize_answer(actual) == normalize_answer(expected))

        all_correct = all_correct and ok
        status = "CORRECT" if ok else "INCORRECT"

        print(f"Output summary: {actual}")
        print(f"Expected summary: {expected}")
        print(f"Result: {status}")

    print("=" * 80)

run_tests()

Test 1
n = 6
|L| = 6
Output summary: [[5, 0], [5, 4, 3, 2, 1, 0, 5]]
Expected summary: [[5, 0], [5, 4, 3, 2, 1, 0, 5]]
Result: CORRECT
Test 2
n = 5
|L| = 3
Output summary: None
Expected summary: None
Result: CORRECT
Test 3
n = 10680
|L| = 24316
Output summary: [[4, 5], [4, 3, 5, 4]]
Expected summary: [[4, 5], [4, 3, 5, 4]]
Result: CORRECT
